# Training parameters for YOLO  for plan 1 (only for black symbols)
## Big plan with rotated symbols
### DPI setting was 500 because of large layouts and pixel error 

In [ ]:


model = YOLO("yolov8l-obb.pt")

model.train(
    data="/content/datasetcombinedbaselinetestobbv31/data.yaml",
    imgsz=1024,
    epochs=120,
    batch=6,                 # T4-safe real batch
    device=0,
    workers=2,               # modest to avoid dataloader OOMs
    seed=0,
    rect=False,              # keep shuffling (you train on square tiles anyway)
    cache="disk",            # deterministic, RAM-friendly

    # Augment (no rotation: OBB + your dataset already encode angles)
    degrees=0.0, translate=0.05, scale=0.20, shear=0.0, perspective=0.0,
    fliplr=0.20, flipud=0.0, mosaic=0.40, mixup=0.10, close_mosaic=10,
    hsv_h=0.005, hsv_s=0.70, hsv_v=0.40,

    # Optim
    optimizer="AdamW", lr0=2e-3, lrf=0.10, weight_decay=5e-4,
    warmup_epochs=3, cos_lr=True, patience=50, amp=True,

    project="runs_obb", name="stageA_1024",save_period=10
)


# Training parameters for YOLO  for plan 1 (for colored symbols also)
## Big plan with rotated symbols

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l-obb.pt")

model.train(
    data="/content/dataset_final/data.yaml",
    imgsz=1024,               # Downsampling for efficient run on CPU
    epochs=120,
    batch=8,                 # T4 friendly batch
    device=0,
    workers=4,               # Reduced from 8
    seed=42,
    cache="disk",

    # HSV - realistic shade/warmth
    hsv_h=0.05,
    hsv_s=0.6,
    hsv_v=0.4,

    # Geometric
    degrees=0.0,
    translate=0.05,
    scale=0.20,
    fliplr=0.20,
    mosaic=0.40,
    mixup=0.10,
    close_mosaic=10,

    # Optimizer
    optimizer="AdamW",
    lr0=1e-3,                # Slightly reduced for smaller batch
    lrf=0.10,
    weight_decay=5e-4,
    warmup_epochs=3,
    cos_lr=True,
    patience=50,
    amp=True,                # Keep AMP enabled - critical for T4

    project="runs_obb_colorfinal",
    name="t4_color_v1",
    exist_ok=True,
)


# Parameters for YOLO prediction 
## Big plan with rotated symbols 

In [ ]:
# =============================================================================
# Big Farbig Track Plans - YOLO Prediction Parameters
# =============================================================================

from ultralytics import YOLO
import cv2

# Load model
model = YOLO("path/to/wien_model.pt")

# Detection parameters
TILE_SIZE = 2048
OVERLAP_PCT = 40
DPI = 500
PRED_IMGSZ = 1024
TILE_HALO = 320


# Test-time augmentation
USE_TTA = True
TTA_SCALES = [1.0]
TTA_FLIPS = [0, 1]
TTA_MIN_VOTES = 1

# Per-class confidence thresholds
CLASS_CONF = {
    "signal": 0.40,
    "gm_block": 0.22,
    "gks_festkodiert": 0.85,
    "gks_gesteuert": 0.50,
    "weichen_block": 0.42,
    "isolierstoß": 0.09,
    "haltepunkt": 0.32,
    "sverbinder": 0.50,
    "coordinate": 0.10,
    "prellbock": 0.30,
    "haltetafel": 0.81,
    "weichenende": 0.70,
    "weichengruppenende": 0.70,
}

# NMS thresholds
NMS_THRESH = {
    "signal": 0.32,
    "gm_block": 0.40,
    "gks_festkodiert": 0.30,
    "gks_gesteuert": 0.30,
    "weichen_block": 0.30,
    "isolierstoß": 0.40,
    "haltepunkt": 0.40,
    "sverbinder": 0.40,
    "coordinate": 0.25,
    "prellbock": 0.40,
    "haltetafel": 0.35,
    "weichenende": 0.40,
    "weichengruppenende": 0.40,
}

# Run prediction
results = model.predict(
    source="path/to/image.png",
    imgsz=PRED_IMGSZ,
    conf=0.09,  # Use lowest class threshold
    iou=0.30,   # Default NMS IoU
    save=True,
    save_txt=True,
    save_conf=True,
    augment=USE_TTA,
)


# Training Parameters for the smaller Plan 
## DPI setting was 800 because of smaller symbols

In [ ]:
from ultralytics import YOLO

# Load YOLOv8l-OBB pretrained model
model = YOLO("yolov8l-obb.pt")

# Train the model
results = model.train(
    # ═══════════════════════════════════════════════════════════════════
    # DATASET CONFIG - CHANGED FOR 1024
    # ═══════════════════════════════════════════════════════════════════
    data="/content/antwerp_1024_final/data.yaml",  # ← CHANGED
    imgsz=1024,              # same as dataet tile size

    # ═══════════════════════════════════════════════════════════════════
    # TRAINING CONFIG - YOUR PROVEN VALUES FROM 1280
    # ═══════════════════════════════════════════════════════════════════
    epochs=120,              # ← YOUR PROVEN VALUE (93% mAP)
    batch=8,                # ← YOUR PROVEN VALUE
    device=0,                # GPU
    workers=4,               # CPU cores
    seed=0,                  # Reproducibility
    rect=False,              # Keep shuffling (square tiles)
    cache="ram",             # Cache in RAM

    # ═══════════════════════════════════════════════════════════════════
    # AUGMENTATION - YOUR PROVEN VALUES FROM 1280
    # ═══════════════════════════════════════════════════════════════════
    # Geometric augmentation
    degrees=0.0,             # NO rotation (already at 90° increments)
    translate=0.05,          # ← YOUR VALUE: 5% shift
    scale=0.20,              # ← YOUR VALUE: 20% zoom (NOT 0.30!)
    shear=0.0,               # No shear
    perspective=0.0,         # No perspective warp
    fliplr=0.20,             # ← YOUR VALUE: 20% horizontal flip (NOT 0.25!)
    flipud=0.0,              # No vertical flip

    # Mosaic/Mixup
    mosaic=0.40,             # ← YOUR VALUE: 40% mosaic (NOT 0.55!)
    mixup=0.10,              # ← YOUR VALUE: 10% mixup (NOT 0.20!)
    close_mosaic=10,         # ← YOUR VALUE: Disable mosaic last 10 epochs (NOT 20!)

    # Color augmentation - DISABLED (semantic colors)
    hsv_h=0.0,               # CRITICAL: NO hue shift
    hsv_s=0.0,               # CRITICAL: NO saturation change
    hsv_v=0.0,               # CRITICAL: NO brightness change

    # ═══════════════════════════════════════════════════════════════════
    # OPTIMIZER - YOUR PROVEN VALUES FROM 1280
    # ═══════════════════════════════════════════════════════════════════
    optimizer="AdamW",
    lr0=2e-3,                # ← YOUR VALUE: 0.002 (NOT 0.0012!)
    lrf=0.10,                # Final LR = lr0 × 0.10
    weight_decay=5e-4,
    warmup_epochs=5,         # ← YOUR VALUE: 5 epochs (NOT 8!)
    cos_lr=True,             # Cosine LR scheduler
    patience=30,             # ← YOUR VALUE: 30 epochs (NOT 70!)
    amp=True,                # Automatic mixed precision (FP16)

    # ═══════════════════════════════════════════════════════════════════
    # OUTPUT
    # ═══════════════════════════════════════════════════════════════════
    project="runs_obb_antwerp",
    name="antwerp_v1_1024_h100",  # ← CHANGED for 1024
    verbose=True,
)

print("\n✓ Training complete!")

# Prediction parameters for the YOLO for the smaller plan 

In [ ]:
# =============================================================================
# Antwerp Railway Schematics - YOLO Prediction Parameters
# =============================================================================

from ultralytics import YOLO
import cv2

# Load model
model = YOLO("path/to/antwerp_model.pt")

# Detection parameters
TILE_SIZE = 1024
OVERLAP_PCT = 60
DPI = 800
PRED_IMGSZ = 1024
TILE_HALO = 100


# Antwerp-specific detection features
GLOBAL_CONF_THRESHOLD = 0.05
USE_INK_FILTER = True
INK_THRESHOLD = 0.012  # 1.2% minimum ink
USE_CENTROID_HALO = True
HALO_RATIO = 0.12
HALO_CONF_BOOST = 0.50
FILTER_CONTAINED_BOXES = True
CONTAINED_BOX_THRESHOLD = 0.80
PREFER_LARGER_NMS = True
USE_NATIVE_OBB_POLYGONS = True
USE_HALO_EXPANSION = True

# Test-time augmentation
USE_TTA = True
TTA_SCALES = [1.0]
TTA_FLIPS = [0, 1]
TTA_MIN_VOTES = 1

# Per-class confidence thresholds
CLASS_CONF = {
    "Signal": 0.25,
    "coordinate": 0.35,
    "text_id": 0.35,
    "mech_point": 0.25,
    "elec_point": 0.20,
    "insulation_joint": 0.30,
    "s_bond": 0.75,
    "short_bond": 0.20,
    "terminal_bond": 0.40,
    "coupling_coil_active": 0.35,
    "coupling_coil_disabled": 0.45,
    "spie_loop": 0.45,
}

# NMS thresholds
NMS_THRESH = {
    "Signal": 0.35,
    "coordinate": 0.30,
    "text_id": 0.30,
    "mech_point": 0.35,
    "elec_point": 0.35,
    "insulation_joint": 0.35,
    "s_bond": 0.35,
    "short_bond": 0.35,
    "terminal_bond": 0.35,
    "coupling_coil_active": 0.35,
    "coupling_coil_disabled": 0.30,
    "spie_loop": 0.30,
}

# Run prediction
results = model.predict(
    source="path/to/image.png",
    imgsz=PRED_IMGSZ,
    conf=GLOBAL_CONF_THRESHOLD,  # 0.05
    iou=0.30,
    save=True,
    save_txt=True,
    save_conf=True,
    augment=USE_TTA,
)
